# Reverse PCA: fit on AoU, project 1000G onto it

Every other notebook fits the PCA on 1000G and projects AoU onto it. This one runs it backwards: fit a PCA on a subsample of AoU's own round-1-passing (EUR-like) cohort (`N_AOU_PCA_SAMPLES`), project the 1000G EUR reference samples onto *that* AoU-defined space, and reproject all round-1-passing AoU samples onto it. One fit now serves what used to be two separate purposes: (1) a cross-check -- does 1000G cluster recognizably in AoU's own PC space, in roughly the same relative arrangement as the 1000G-onto-1000G picture from `round1_filter.ipynb`/`round2_filter.ipynb`; (2) "round 2b" -- refit the Mahalanobis ellipsoid filter (as in `round2_filter.ipynb`) in this AoU-fit direction, producing an independent keep-list and PC covariates for `residualize_phenotypes.ipynb`, for comparison against the 1000G-fit round 2 result.

Previously this ran as two separate pipelines -- an ancestry-unrestricted, non-HWE-filtered PCA (for the cross-check) and a second, round-1-restricted, HWE-filtered PCA (for round 2b) -- duplicating the HWE filter, LD pruning, PCA fit, and both projections. Unified into one: since the restricted+HWE-filtered fit is the one actually trusted for the ellipsoid filter and downstream covariates, the cross-check plots and loadings diagnostics below are now generated from that same fit rather than a separate ancestry-unrestricted one.

LD pruning starts from `agreeing_snps.ids` (round 1's already ID+REF+ALT-verified, HM3-restricted variant set) rather than AoU's full variant catalog -- keeps the AoU-fit PCA and the reference projection on a variant set already known to cross-match cleanly, without needing a second harmonization pass.

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

## Inputs

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/01_ancestry_filtering"

# from round 1 (submit_pca_r1.ipynb)
OUT_PREFIX = f"{BUCKET_DIR}/1kg_all_hm3"                                    # QC'd, whole-cohort 1000G bfile
PANEL_PATH = f"{BUCKET_DIR}/integrated_call_samples_v3.20130502.ALL.panel"
ACAF_BIALLELIC_PREFIX = f"{BUCKET_DIR}/round1_pca/acaf_hm3_subset_biallelic"  # already biallelic-filtered, HM3-restricted ACAF
AGREEING_SNPS_PATH = f"{BUCKET_DIR}/round1_pca/agreeing_snps.ids"            # ID+REF+ALT-verified vs the reference

# from round 1 filtering (round1_filter.ipynb) -- prob_tag there is f"p{THRESHOLD_QUANTILE*100:g}",
# so THRESHOLD_QUANTILE=0.999 renders as "p99.9", not "p999" -- confirm this matches what that
# notebook's OUT_DIR + prob_tag actually produced before running
ROUND1_KEEP_PATH = f"{BUCKET_DIR}/round1_pca/round1_keep_ids_eur_p99.9.txt"

EUR_REF_POPS = ["CEU", "GBR", "TSI", "FIN", "IBS"]   # full EUR reference pool -- cross-check plot + biplots
ELLIPSOID_REF_POPS = ["CEU", "GBR"]                  # tighter cluster the Mahalanobis ellipsoid is fit to, matching round2_filter.ipynb's REF_POPS

N_AOU_PCA_SAMPLES = 10000       # drawn from ROUND1_KEEP_PATH, fits the AoU-side PCA
N_PCS = 2                       # Mahalanobis fit dimensionality, matching round2_filter.ipynb's N_PCS
THRESHOLD_QUANTILE = 0.999999   # round 2b's threshold -- matches round2_filter.ipynb's tighter (vs round 1's 0.999) refit
RANDOM_SEED = 0
N_PCS_TO_PLOT = 6                # scree / Manhattan / biplot panel count
N_TOP_LOADINGS = 20
MAX_AOU_POINTS = 20000            # biplot subsample cap, matching pc_biplots.ipynb

OUT_DIR = f"{BUCKET_DIR}/reverse_pca_aou"
os.makedirs(OUT_DIR, exist_ok=True)

AOU_SAMPLE_PATH = os.path.join(OUT_DIR, "aou_pca_sample.ids")            # N_AOU_PCA_SAMPLES drawn from ROUND1_KEEP_PATH
PRUNE_PREFIX = os.path.join(OUT_DIR, "aou_pruned")
HWE_SNPLIST = os.path.join(OUT_DIR, "aou_hwe_filtered.snplist")
PCA_PREFIX = os.path.join(OUT_DIR, "aou_pca")
REF_KEEP_PATH = os.path.join(OUT_DIR, "eur_ref_keep.ids")
REF_PROJECT_PREFIX = os.path.join(OUT_DIR, "ref_projected_onto_aou")
AOU_ALL_REPROJECT_PREFIX = os.path.join(OUT_DIR, "aou_all_reprojected")  # ALL round-1 passers, not just the fit subsample

print(OUT_DIR)

## Sample AoU individuals for the PCA fit (round-1 passers)

In [ ]:
import pandas as pd

round1_ids = pd.read_csv(ROUND1_KEEP_PATH, header=None)[0]
n_fit = min(N_AOU_PCA_SAMPLES, len(round1_ids))
aou_sample_ids = round1_ids.sample(n=n_fit, random_state=RANDOM_SEED)
aou_sample_ids.to_csv(AOU_SAMPLE_PATH, index=False, header=False)
print(f"Sampled {len(aou_sample_ids)} of {len(round1_ids)} round-1-passing AoU individuals -> {AOU_SAMPLE_PATH}")

## HWE filter, LD prune, PCA -- fit on the AoU subsample

This subsample is already EUR-like (round 1 passers), so an HWE step is appropriate here -- same reasoning as `submit_pca_r2.ipynb`'s within-EUR-pool HWE step. `--hwe 1e-5 0.001 keep-fewhet`: `k=0.001` scales the threshold to this subsample's size (plink2's documented recommendation; the flat default trips plink2's "suspiciously strict" error at this scale), `keep-fewhet` only excludes excess-heterozygosity variants so residual structure within this still-not-perfectly-homogeneous EUR-like cohort isn't mistaken for genotyping error. `--nonfounders` throughout for the same reason as everywhere else in this pipeline.

In [ ]:
%%bash -s "$ACAF_BIALLELIC_PREFIX" "$AGREEING_SNPS_PATH" "$AOU_SAMPLE_PATH" "$HWE_SNPLIST" "$PRUNE_PREFIX" "$PCA_PREFIX"
set -e
ACAF_BIALLELIC_PREFIX=$1
AGREEING_SNPS_PATH=$2
AOU_SAMPLE_PATH=$3
HWE_SNPLIST=$4
PRUNE_PREFIX=$5
PCA_PREFIX=$6

HWE_PREFIX="${PRUNE_PREFIX}_hwe"

plink2 \
  --pfile "$ACAF_BIALLELIC_PREFIX" \
  --keep "$AOU_SAMPLE_PATH" \
  --extract "$AGREEING_SNPS_PATH" \
  --nonfounders \
  --hwe 1e-5 0.001 keep-fewhet \
  --write-snplist \
  --out "$HWE_PREFIX"

cp "${HWE_PREFIX}.snplist" "$HWE_SNPLIST"
wc -l "$HWE_SNPLIST"

plink2 \
  --pfile "$ACAF_BIALLELIC_PREFIX" \
  --keep "$AOU_SAMPLE_PATH" \
  --extract "$HWE_SNPLIST" \
  --nonfounders \
  --indep-pairwise 200kb 1 0.1 \
  --out "$PRUNE_PREFIX"

wc -l "${PRUNE_PREFIX}.prune.in"

plink2 \
  --pfile "$ACAF_BIALLELIC_PREFIX" \
  --keep "$AOU_SAMPLE_PATH" \
  --extract "${PRUNE_PREFIX}.prune.in" \
  --nonfounders \
  --freq counts \
  --pca allele-wts 20 \
  --out "$PCA_PREFIX"

ls -lh "${PCA_PREFIX}".*

## Reference EUR keep-list

In [ ]:
panel = pd.read_csv(PANEL_PATH, sep="\t")
ref_keep_ids = panel.loc[panel["pop"].isin(EUR_REF_POPS), "sample"]
ref_keep_ids.to_csv(REF_KEEP_PATH, index=False, header=False)
print(f"{len(ref_keep_ids)} reference samples ({'+'.join(EUR_REF_POPS)}) -> {REF_KEEP_PATH}")

## Project the 1000G EUR reference onto the AoU-fit PCs

AVG scoring (no `sum` modifier) and `list-variants`, same conventions as every other projection in this pipeline -- both sides of the comparisons below are `--score` output scored identically, and `list-variants` records the exact variant set this run actually used for the reprojection below to reuse.

In [ ]:
%%bash -s "$OUT_PREFIX" "$PCA_PREFIX" "$PRUNE_PREFIX" "$REF_KEEP_PATH" "$REF_PROJECT_PREFIX"
set -e
OUT_PREFIX=$1
PCA_PREFIX=$2
PRUNE_PREFIX=$3
REF_KEEP_PATH=$4
REF_PROJECT_PREFIX=$5

WEIGHTS="${PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")
ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC20' | cut -d: -f1)

plink2 \
  --bfile "$OUT_PREFIX" \
  --keep "$REF_KEEP_PATH" \
  --extract "${PRUNE_PREFIX}.prune.in" \
  --nonfounders \
  --read-freq "${PCA_PREFIX}.acount" \
  --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize list-variants \
  --score-col-nums "${PC1_COL}-${PC_LAST_COL}" \
  --out "$REF_PROJECT_PREFIX"

head "${REF_PROJECT_PREFIX}.sscore"
wc -l "${REF_PROJECT_PREFIX}.sscore.vars"

## Reproject ALL round-1-passing AoU samples onto the AoU-fit PCs

`--keep "$ROUND1_KEEP_PATH"`, not `AOU_SAMPLE_PATH` -- the full round-1-passing cohort, not just the `N_AOU_PCA_SAMPLES` subset the PCA was fit on. This is the table the sanity check, cross-check plot, biplots, Mahalanobis filter, and final PC covariates all draw from. Restricted to `${REF_PROJECT_PREFIX}.sscore.vars` (the `list-variants` output above) -- the exact variant set that run actually used, same lesson as everywhere else in this pipeline.

In [ ]:
%%bash -s "$ACAF_BIALLELIC_PREFIX" "$ROUND1_KEEP_PATH" "$PCA_PREFIX" "$REF_PROJECT_PREFIX" "$AOU_ALL_REPROJECT_PREFIX"
set -e
ACAF_BIALLELIC_PREFIX=$1
ROUND1_KEEP_PATH=$2
PCA_PREFIX=$3
REF_PROJECT_PREFIX=$4
AOU_ALL_REPROJECT_PREFIX=$5

USED_VARS="${REF_PROJECT_PREFIX}.sscore.vars"
if [ ! -s "$USED_VARS" ]; then
  echo "Missing $USED_VARS -- run the reference projection cell (with 'list-variants') first" >&2
  exit 1
fi

WEIGHTS="${PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")
ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC20' | cut -d: -f1)

plink2 \
  --pfile "$ACAF_BIALLELIC_PREFIX" \
  --keep "$ROUND1_KEEP_PATH" \
  --extract "$USED_VARS" \
  --nonfounders \
  --read-freq "${PCA_PREFIX}.acount" \
  --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize \
  --score-col-nums "${PC1_COL}-${PC_LAST_COL}" \
  --out "$AOU_ALL_REPROJECT_PREFIX"

head "${AOU_ALL_REPROJECT_PREFIX}.sscore"
wc -l "${AOU_ALL_REPROJECT_PREFIX}.sscore"

## Sanity check: reprojection recovers the direct PCA fit

Restricts the reprojected-AoU table to just the `AOU_SAMPLE_PATH` fit subsample and compares against the direct `--pca` eigenvectors -- should closely agree (up to arbitrary per-PC sign). Same check used everywhere else in this pipeline.

In [ ]:
direct = pd.read_csv(f"{PCA_PREFIX}.eigenvec", sep=r"\s+")
reproj_all = pd.read_csv(f"{AOU_ALL_REPROJECT_PREFIX}.sscore", sep=r"\s+")

direct_id_col = "#IID" if "#IID" in direct.columns else "IID"
reproj_id_col = "#IID" if "#IID" in reproj_all.columns else "IID"

fit_ids = set(pd.read_csv(AOU_SAMPLE_PATH, header=None)[0])
reproj_fit = reproj_all[reproj_all[reproj_id_col].isin(fit_ids)]

merged = direct.merge(reproj_fit, left_on=direct_id_col, right_on=reproj_id_col, suffixes=("_direct", "_reproj"))

score_cols = [c for c in reproj_all.columns if c.startswith("PC") and c.endswith("_AVG")]
for i, pc in enumerate(range(1, 21)):
    direct_col = f"PC{pc}"
    score_col = score_cols[i] if i < len(score_cols) else None
    if direct_col in merged.columns and score_col in merged.columns:
        corr = merged[direct_col].corr(merged[score_col])
        print(f"PC{pc}: r = {corr:.4f}")

## Plot: reference EUR populations projected into AoU's own PC space

Same population palette/marker convention as `round1_filter.ipynb`/`round2_filter.ipynb`/`pc_biplots.ipynb`. Only the two `--score`-derived layers are plotted -- all round-1-passing AoU samples' reprojected scores (background, grey) and the 1000G EUR reference's projected scores (foreground, full alpha) -- not the AoU-fit PCA's own direct `.eigenvec` output, so both layers are on the same `--score`/AVG-scale footing. `ref_proj` and `aou_reproj` built here are reused by the biplots and the Mahalanobis filter below.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

SUPERPOP_POPS = {
    "EUR": ["CEU", "TSI", "FIN", "GBR", "IBS"],
    "AFR": ["YRI", "LWK", "GWD", "MSL", "ESN", "ASW", "ACB"],
    "EAS": ["CHB", "JPT", "CHS", "CDX", "KHV"],
    "SAS": ["GIH", "PJL", "BEB", "STU", "ITU"],
    "AMR": ["MXL", "PUR", "CLM", "PEL"],
}
SUPERPOP_CMAPS = {"EUR": "Blues", "AFR": "Oranges", "EAS": "Greens", "SAS": "Purples", "AMR": "Reds"}
SUPERPOP_SHADE_RANGES = {
    "EUR": (0.25, 0.95), "AFR": (0.35, 0.90), "EAS": (0.30, 0.95), "SAS": (0.25, 0.85), "AMR": (0.35, 0.90),
}
POP_MARKERS_CYCLE = ["o", "s", "^", "D", "v", "P", "X", "*"]


def build_pop_colors(superpop_pops=SUPERPOP_POPS, superpop_cmaps=SUPERPOP_CMAPS, shade_ranges=SUPERPOP_SHADE_RANGES):
    colors = {}
    for superpop, pops in superpop_pops.items():
        cmap = matplotlib.colormaps[superpop_cmaps[superpop]]
        lo, hi = shade_ranges[superpop]
        shades = np.linspace(lo, hi, len(pops)) if len(pops) > 1 else [(lo + hi) / 2]
        for pop, frac in zip(pops, shades):
            colors[pop] = mcolors.to_hex(cmap(frac))
    return colors


def build_pop_markers(superpop_pops=SUPERPOP_POPS, markers_cycle=POP_MARKERS_CYCLE):
    markers = {}
    for pops in superpop_pops.values():
        for i, pop in enumerate(pops):
            markers[pop] = markers_cycle[i % len(markers_cycle)]
    return markers


POP_COLORS = build_pop_colors()
POP_MARKERS = build_pop_markers()

aou_reproj = reproj_all.rename(columns={reproj_id_col: "sample"})

ref_proj = pd.read_csv(f"{REF_PROJECT_PREFIX}.sscore", sep=r"\s+")
ref_id_col = "#IID" if "#IID" in ref_proj.columns else "IID"
ref_proj = ref_proj.rename(columns={ref_id_col: "sample"})
ref_proj = ref_proj.merge(panel[["sample", "pop", "super_pop"]], on="sample")

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(
    aou_reproj["PC1_AVG"], aou_reproj["PC2_AVG"],
    c="grey", marker="x", alpha=0.3, s=10, label="AoU (round-1-passing, reprojected)",
)
for pop, grp in ref_proj.groupby("pop"):
    ax.scatter(
        grp["PC1_AVG"], grp["PC2_AVG"],
        c=POP_COLORS.get(pop, "black"), marker=POP_MARKERS.get(pop, "o"),
        label=pop, alpha=1.0, s=15,
    )
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("1000G EUR reference projected into AoU's own PC space")
ax.legend(markerscale=2, fontsize=8)
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "reverse_pca_projection.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Pairwise PC biplots (SPLOM), 1000G projected on

Same convention as `pc_biplots.ipynb` -- lower-triangle scatterplot matrix across `PC1..N_PCS_TO_PLOT`, reference (1000G EUR, all `EUR_REF_POPS`) colored by population, AoU (round-1-passing, reprojected) as a background cloud subsampled to `MAX_AOU_POINTS`. Diagonal shows each PC's marginal distribution (reference vs AoU) instead of a scatter against itself. Reuses `ref_proj`/`aou_reproj` from the cell above rather than reloading -- same `--score` output, just more PC pairs.

In [ ]:
pc_cols_biplot = [f"PC{i}_AVG" for i in range(1, N_PCS_TO_PLOT + 1)]
assert all(c in ref_proj.columns for c in pc_cols_biplot) and all(c in aou_reproj.columns for c in pc_cols_biplot), \
    "missing PC columns for biplot -- check N_PCS_TO_PLOT against the PCA's actual PC count"

if len(aou_reproj) > MAX_AOU_POINTS:
    aou_biplot = aou_reproj.sample(n=MAX_AOU_POINTS, random_state=RANDOM_SEED)
else:
    aou_biplot = aou_reproj
print(f"Plotting {len(aou_biplot)} AoU points (subsampled from {len(aou_reproj)})")

n = len(pc_cols_biplot)
fig, axes = plt.subplots(n, n, figsize=(2.2 * n, 2.2 * n))

for row in range(n):
    for col in range(n):
        ax = axes[row, col]
        if row < col:
            ax.axis("off")
            continue

        y_col, x_col = pc_cols_biplot[row], pc_cols_biplot[col]

        if row == col:
            ax.hist(ref_proj[x_col], bins=40, density=True, color="grey", alpha=0.6, label="1000G")
            ax.hist(aou_biplot[x_col], bins=40, density=True, color="royalblue", alpha=0.4, label="AoU")
            if row == 0:
                ax.legend(fontsize=6, loc="upper right")
        else:
            ax.scatter(
                aou_biplot[x_col], aou_biplot[y_col],
                c="grey", s=3, alpha=0.3, linewidths=0, rasterized=True,
            )
            for pop, grp in ref_proj.groupby("pop"):
                ax.scatter(
                    grp[x_col], grp[y_col],
                    c=POP_COLORS.get(pop, "black"), marker=POP_MARKERS.get(pop, "o"),
                    s=6, alpha=1.0, linewidths=0, rasterized=True,
                )

        if row == n - 1:
            ax.set_xlabel(x_col, fontsize=8)
        else:
            ax.set_xticklabels([])
        if col == 0:
            ax.set_ylabel(y_col, fontsize=8)
        else:
            ax.set_yticklabels([])
        ax.tick_params(labelsize=6)

handles = [
    plt.Line2D([0], [0], marker=POP_MARKERS.get(pop, "o"), linestyle="", color=POP_COLORS[pop], label=pop)
    for pop in sorted(ref_proj["pop"].unique())
]
handles.append(plt.Line2D([0], [0], marker="o", linestyle="", color="grey", label="AoU"))
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=7, title="Population")

plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "reverse_pca_biplots.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Loadings diagnostics: scree plot + Manhattan plot

Same method as `pca_loadings_r2.ipynb`, applied to this AoU-fit PCA -- the same fit that produced the projections plotted above -- checks whether the leading PCs carry real signal (scree plot) and whether any PC is dominated by a handful of SNPs in one genomic region rather than genome-wide structure (Manhattan plot + top-N table).

In [ ]:
eigenval = pd.read_csv(f"{PCA_PREFIX}.eigenval", header=None, names=["eigenvalue"])
eigenval["pc"] = range(1, len(eigenval) + 1)
eigenval["pct_variance"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(eigenval["pc"], eigenval["pct_variance"], color="royalblue")
ax.set_xlabel("PC")
ax.set_ylabel("% variance explained")
ax.set_title("Reverse PCA (AoU-fit) scree plot")
ax.set_xticks(eigenval["pc"])
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "reverse_pca_scree.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")
print(eigenval[["pc", "eigenvalue", "pct_variance"]].to_string(index=False))

In [ ]:
weights_path = f"{PCA_PREFIX}.eigenvec.allele"
weights = pd.read_csv(weights_path, sep="\t")

id_col = "ID"
a1_col = "A1"
pc_cols_raw = [f"PC{i}" for i in range(1, N_PCS_TO_PLOT + 1)]
assert id_col in weights.columns and a1_col in weights.columns, f"unexpected columns: {list(weights.columns)}"
assert all(c in weights.columns for c in pc_cols_raw), f"missing PC columns, have: {list(weights.columns)}"

id_parts = weights[id_col].str.split(":", expand=True)
weights["chr"] = id_parts[0]
weights["pos"] = id_parts[1].astype(int)
weights["alt"] = id_parts[3]

loadings = weights[weights[a1_col] == weights["alt"]].copy()
print(f"{len(weights)} total rows -> {len(loadings)} one-row-per-variant (ALT-allele) loadings")


def chr_sort_key(c):
    c = c.replace("chr", "")
    return int(c) if c.isdigit() else 100


chr_order = sorted(loadings["chr"].unique(), key=chr_sort_key)
chr_offset = {}
offset = 0
for c in chr_order:
    chr_offset[c] = offset
    offset += loadings.loc[loadings["chr"] == c, "pos"].max() + 1
loadings["plot_x"] = loadings.apply(lambda r: chr_offset[r["chr"]] + r["pos"], axis=1)

chr_colors = {c: ("#4a4a4a" if i % 2 == 0 else "#a0a0a0") for i, c in enumerate(chr_order)}

for pc in pc_cols_raw:
    fig, ax = plt.subplots(figsize=(12, 3))
    for c in chr_order:
        sub = loadings[loadings["chr"] == c]
        ax.scatter(sub["plot_x"], sub[pc], s=4, color=chr_colors[c], alpha=0.6, rasterized=True)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_ylabel(f"{pc} loading")
    ax.set_xlabel("Genomic position (chromosomes concatenated)")
    ax.set_title(f"Reverse PCA (AoU-fit) {pc} loadings by position")
    plt.tight_layout()
    plot_path = os.path.join(OUT_DIR, f"reverse_pca_loadings_{pc}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {plot_path}")

    top = loadings.reindex(loadings[pc].abs().sort_values(ascending=False).index).head(N_TOP_LOADINGS)
    print(f"\nTop {N_TOP_LOADINGS} |{pc}| loadings:")
    print(top[["ID", "chr", "pos", pc]].to_string(index=False))
    print()

## Mahalanobis ellipsoid filter (round 2b)

Same method as `round2_filter.ipynb` -- fit on the reprojected 1000G reference (`ELLIPSOID_REF_POPS`, CEU/GBR), score all reprojected round-1-passing AoU (`ref_proj`/`aou_reproj` from above) against it, write the round 2b keep-list. `THRESHOLD_QUANTILE` matches `round2_filter.ipynb`'s now-decided tighter threshold.

In [ ]:
from scipy.stats import chi2
from matplotlib.patches import Ellipse

pc_cols_mahal = [f"PC{i}_AVG" for i in range(1, N_PCS + 1)]
assert all(c in ref_proj.columns for c in pc_cols_mahal), f"missing PC columns, have: {list(ref_proj.columns)}"
assert all(c in aou_reproj.columns for c in pc_cols_mahal), f"missing PC columns, have: {list(aou_reproj.columns)}"

print(f"Reference ({'+'.join(ELLIPSOID_REF_POPS)}): {len(ref_proj)} samples")
print(f"AoU (all round-1-passing, reprojected): {len(aou_reproj)} samples")


def mahal(x, mean, cov_inv):
    d = x - mean
    return np.sqrt(d @ cov_inv @ d)


def run_ellipsoid_filter(cluster_name, ref_pop_list):
    ref_mask = ref_proj["pop"].isin(ref_pop_list)
    ref_pcs = ref_proj.loc[ref_mask, pc_cols_mahal].values
    assert len(ref_pcs) > N_PCS, f"too few reference samples ({len(ref_pcs)}) to fit a {N_PCS}-PC covariance"

    ref_mean = ref_pcs.mean(axis=0)
    ref_cov = np.cov(ref_pcs, rowvar=False)
    ref_cov_inv = np.linalg.inv(ref_cov)
    threshold = np.sqrt(chi2.ppf(THRESHOLD_QUANTILE, df=N_PCS))
    print(f"[{cluster_name}] Mahalanobis threshold ({N_PCS} PCs, {THRESHOLD_QUANTILE:.4%}): {threshold:.3f}")

    prob_tag = f"p{THRESHOLD_QUANTILE * 100:g}"
    mahal_col, pass_col = f"mahal_{cluster_name}", f"pass_{cluster_name}"

    ref_proj[mahal_col] = [mahal(row, ref_mean, ref_cov_inv) for row in ref_proj[pc_cols_mahal].values]
    ref_proj[pass_col] = ref_proj[mahal_col] <= threshold

    print(f"[{cluster_name}] Pass rate by population:")
    summary = ref_proj.groupby("pop")[pass_col].agg(["sum", "count"])
    summary["pct"] = (summary["sum"] / summary["count"] * 100).round(1)
    print(summary.sort_values("pct", ascending=False))

    ref_retention = ref_proj.loc[ref_mask, pass_col].mean()
    print(f"[{cluster_name}] {'+'.join(ref_pop_list)} retention: {ref_retention:.1%}")

    aou_reproj[mahal_col] = [mahal(row, ref_mean, ref_cov_inv) for row in aou_reproj[pc_cols_mahal].values]
    aou_reproj[pass_col] = aou_reproj[mahal_col] <= threshold
    n_pass = aou_reproj[pass_col].sum()
    print(f"[{cluster_name}] AoU passing: {n_pass} ({n_pass / len(aou_reproj):.1%})")

    keep_path = os.path.join(OUT_DIR, f"round2b_keep_ids_{cluster_name}_{prob_tag}.txt")
    aou_reproj.loc[aou_reproj[pass_col], "sample"].to_csv(keep_path, index=False, header=False)
    print(f"[{cluster_name}] Wrote {n_pass} IDs to {keep_path}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].scatter(
        aou_reproj[pc_cols_mahal[0]], aou_reproj[pc_cols_mahal[1]],
        c=np.where(aou_reproj[pass_col], "black", "grey"),
        marker="x", alpha=0.3, s=10, label="AoU (round 1 pass, reprojected)",
    )
    for pop, grp in ref_proj.groupby("pop"):
        axes[0].scatter(
            grp[pc_cols_mahal[0]], grp[pc_cols_mahal[1]],
            c=POP_COLORS.get(pop, "grey"), marker=POP_MARKERS.get(pop, "o"),
            label=pop, alpha=1.0, s=15,
        )

    cov_2d = np.cov(ref_pcs[:, :2], rowvar=False)
    mean_2d = ref_pcs[:, :2].mean(axis=0)
    eigvals, eigvecs = np.linalg.eigh(cov_2d)
    angle = np.degrees(np.arctan2(*eigvecs[:, 1][::-1]))
    scale_2d = np.sqrt(chi2.ppf(THRESHOLD_QUANTILE, df=2))
    width = 2 * scale_2d * np.sqrt(eigvals[1])
    height = 2 * scale_2d * np.sqrt(eigvals[0])
    ellipse = Ellipse(
        xy=mean_2d, width=width, height=height, angle=angle,
        fill=False, edgecolor="black", linewidth=2, linestyle="--",
        label=f"{THRESHOLD_QUANTILE:.4%} {'+'.join(ref_pop_list)} ellipsoid (2D slice)",
    )
    axes[0].add_patch(ellipse)
    axes[0].set_xlabel(pc_cols_mahal[0])
    axes[0].set_ylabel(pc_cols_mahal[1])
    axes[0].set_title(f"{pc_cols_mahal[0]} vs {pc_cols_mahal[1]} -- round 2b {cluster_name} reference + AoU")
    axes[0].legend(markerscale=2, fontsize=8)

    for pop, grp in ref_proj.groupby("pop"):
        axes[1].hist(grp[mahal_col], bins=30, alpha=0.5, label=pop, color=POP_COLORS.get(pop, "grey"), density=True)
    axes[1].hist(aou_reproj[mahal_col], bins=30, alpha=0.4, label="AoU", color="royalblue", density=True)
    axes[1].axvline(threshold, color="black", linestyle="--", linewidth=2, label=f"Threshold ({threshold:.2f})")
    axes[1].set_xlabel(f"Mahalanobis distance from {'+'.join(ref_pop_list)} centroid ({N_PCS} PCs)")
    axes[1].set_ylabel("Density")
    axes[1].legend(fontsize=8)

    plt.suptitle(
        f"Round 2b ancestry filter [{cluster_name}] -- {'+'.join(ref_pop_list)} retention: {ref_retention:.1%}  |  "
        f"AoU pass rate: {n_pass / len(aou_reproj):.1%}",
        fontsize=11,
    )
    plt.tight_layout()
    plot_path = os.path.join(OUT_DIR, f"round2b_filter_{cluster_name}_{prob_tag}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[{cluster_name}] Plot saved to {plot_path}")

    return {"mean": ref_mean, "cov_inv": ref_cov_inv, "threshold": threshold, "keep_path": keep_path, "pass_col": pass_col}

In [ ]:
r2b_result = run_ellipsoid_filter("aou_fit_r2b", ELLIPSOID_REF_POPS)

## Save PC covariates for the retained set

Writes `PC1..PC20` for every AoU sample that passed the round 2b ellipsoid filter above, in the `IID PC1 ... PC20` format `residualize_phenotypes.ipynb`'s `PC_PATH` / `pull_covariates()` expects -- these become the PC covariates for phenotype residualization.

In [ ]:
retained = aou_reproj.loc[aou_reproj[r2b_result["pass_col"]]].copy()

pc_avg_cols = [f"PC{i}_AVG" for i in range(1, 21)]
assert all(c in retained.columns for c in pc_avg_cols), f"missing PC columns, have: {list(retained.columns)}"

covariate_table = retained[["sample"] + pc_avg_cols].rename(
    columns={"sample": "IID", **{f"PC{i}_AVG": f"PC{i}" for i in range(1, 21)}}
)

prob_tag = f"p{THRESHOLD_QUANTILE * 100:g}"
PC_COVARIATE_PATH = os.path.join(OUT_DIR, f"round2b_pc_covariates_{prob_tag}.txt")
covariate_table.to_csv(PC_COVARIATE_PATH, sep="\t", index=False)
print(f"Wrote {len(covariate_table)} samples' PC1-PC20 -> {PC_COVARIATE_PATH}")